# Setup and Installation

## Import Required Libraries
Loading essential libraries for data manipulation (pandas), Fabric workspace integration (sempy), and AI-powered data agent capabilities (FabricOpenAI)

In [ ]:
import pandas as pd
import time
import re
import os
import sempy.fabric as fabric 
from fabric.dataagent.client import FabricOpenAI
from fabric.dataagent.client import delete_data_agent
from datetime import datetime
from pyspark.sql.functions import lit

## Import Parameters

In [ ]:
%run 2. Parameters

## Analyzing Semantic Model M-expression column dependencies

In [ ]:
df = spark.sql(f"SELECT * FROM {lakehouse_name}.calcdependency")
df = df.toPandas()

# Rename specific columns
df = df.rename(columns={
    'DATABASE_NAME': 'DATABASE_NAME',
    'OBJECT': 'SEMANTIC_MODEL_OBJECT',
    'TABLE': 'SEMANTIC_MODEL_TABLE',
    'REFERENCED_OBJECT': 'SEMANTIC_MODEL_REFERENCED_OBJECT'
})

# Filter rows and select columns
filtered_df = df.loc[
    df['REFERENCED_OBJECT_TYPE'].isin(['M_EXPRESSION', 'PARTITION']),
    ['DATABASE_NAME', 'SEMANTIC_MODEL_TABLE', 'SEMANTIC_MODEL_OBJECT', 'EXPRESSION', 'SEMANTIC_MODEL_REFERENCED_OBJECT', 'REFERENCED_EXPRESSION','REFERENCED_OBJECT_TYPE']
]

display(filtered_df)

## Aggregate Dependencies by Semantic Model Object
Grouping calculated columns by semantic model and object name, then consolidating all related dependencies and expressions into comma-separated lists for easier analysis

In [ ]:
grouped_df = (
    filtered_df.groupby(['DATABASE_NAME','SEMANTIC_MODEL_OBJECT'], as_index=False)
    .agg(lambda x: ' , '.join(x.astype(str).unique()))
)

display(grouped_df)

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.calcdependency LIMIT 1000")
display(df)

# Extract Semantic Model Metadata

## Extract Source Schema and Table Information
Parsing M Expression syntax to extract the underlying source schema and table/view names from the expression strings, enabling better understanding of data lineage and dependencies

In [ ]:
def extract_schema_table_or_view(text):
    # default values if not found
    schema = None
    item = None
    
    # handle None/NaN values
    if pd.isna(text) or text is None:
        return pd.Series([schema, item], index=['SCHEMA', 'TABLE_OR_VIEW'])

    # convert to string to be safe
    text = str(text)
    
    # find Schema=
    if 'Schema="' in text:
        part = text.split('Schema="', 1)[1]
        schema = part.split('"', 1)[0]
    
    # find Item (Table or View)= - handle both single and double quotes
    if 'Item=""' in text:
        part = text.split('Item=""', 1)[1]
        item = part.split('""', 1)[0]
    elif 'Item="' in text:
        part = text.split('Item="', 1)[1]
        item = part.split('"', 1)[0]
    
    return pd.Series([schema, item], index=['SCHEMA', 'TABLE_OR_VIEW'])

# apply to the dataframe
grouped_df[['SCHEMA', 'TABLE_OR_VIEW']] = grouped_df['EXPRESSION'].apply(extract_schema_table_or_view)

# Check if SCHEMA or TABLE_OR_VIEW is null, then apply to REFERENCED_EXPRESSION
mask = grouped_df['SCHEMA'].isna() | grouped_df['TABLE_OR_VIEW'].isna()
grouped_df.loc[mask, ['SCHEMA', 'TABLE_OR_VIEW']] = grouped_df.loc[mask, 'REFERENCED_EXPRESSION'].apply(extract_schema_table_or_view)

# Display selected columns
columns_to_show = ['DATABASE_NAME', 'SEMANTIC_MODEL_TABLE', 'SEMANTIC_MODEL_OBJECT', 'SEMANTIC_MODEL_REFERENCED_OBJECT', 'SCHEMA', 'TABLE_OR_VIEW',
'REFERENCED_OBJECT_TYPE','EXPRESSION', 'REFERENCED_EXPRESSION']

display(grouped_df[columns_to_show])

## Identify Data Source Types
Detecting the origin of each data source by analyzing expression patterns to classify sources as SharePoint, Serverless SQL, or Table references for comprehensive source tracking

In [ ]:
def detect_source(text):
    # handle None/NaN values
    if pd.isna(text) or text is None:
        return None
    
    # convert to string to be safe
    text = str(text)
    
    if 'SharePoint.Files' in text:
        return 'SharePoint'
    elif 'Sql.Database' in text:
        return 'Database'
    elif 'Csv.' in text:
        return 'Csv'
    elif 'Folder.Files' in text:
        return 'Folder'  
    elif 'Table.' in text:
        return 'Semantic Model Table'      
    else:
        return None


# Apply to the dataframe for SOURCE_TYPE
grouped_df['SOURCE_TYPE'] = grouped_df['EXPRESSION'].apply(detect_source)

# For rows where SOURCE_TYPE is 'Table', check REFERENCED_EXPRESSION for SharePoint or Serverless
mask_table = grouped_df['SOURCE_TYPE'] == 'Table'
grouped_df.loc[mask_table, 'SOURCE_TYPE'] = grouped_df.loc[mask_table, 'REFERENCED_EXPRESSION'].apply(
    lambda x: 'SharePoint' if (pd.notna(x) and 'SharePoint.Files' in str(x)) 
              else ('Serverless' if (pd.notna(x) and 'Sql.Database' in str(x)) 
              else 'Table')
)

# For rows where SOURCE_TYPE is still null, check REFERENCED_EXPRESSION
mask_source = grouped_df['SOURCE_TYPE'].isna()
grouped_df.loc[mask_source, 'SOURCE_TYPE'] = grouped_df.loc[mask_source, 'REFERENCED_EXPRESSION'].apply(detect_source)

# Display selected columns
columns_to_show = ['DATABASE_NAME', 'SEMANTIC_MODEL_OBJECT', 'SEMANTIC_MODEL_TABLE', 'SEMANTIC_MODEL_REFERENCED_OBJECT', 'SCHEMA', 'TABLE_OR_VIEW', 'SOURCE_TYPE',
'REFERENCED_OBJECT_TYPE','EXPRESSION', 'REFERENCED_EXPRESSION']

display(grouped_df[columns_to_show])

## Extract Connection Strings
Retrieving connection string information from expressions based on source type - extracting server details for Serverless SQL sources and SharePoint URLs for SharePoint sources

In [ ]:
def extract_connection_string(row):
    # ----- Case 1: Serverless -----
    if row['SOURCE_TYPE'] == 'Database':
        text = row.get('REFERENCED_EXPRESSION', '')
        # Check if it starts with 'let' - if yes, return None
        if text.strip().startswith('let'):
            return None
        # Otherwise, extract text inside first quotes
        if '"' in text:
            return text.split('"', 2)[1]
    
    # ----- Case 2: SharePoint ----- extract complete file path
    elif row['SOURCE_TYPE'] == 'SharePoint':
        # First check EXPRESSION for 'SharePoint.Files'
        expression_text = row.get('EXPRESSION', '')
        if 'SharePoint.Files' in expression_text:
            return extract_sharepoint_details(expression_text)
        
        # If not found in EXPRESSION, check REFERENCED_EXPRESSION
        referenced_text = row.get('REFERENCED_EXPRESSION', '')
        if 'SharePoint.Files' in referenced_text:
            return extract_sharepoint_details(referenced_text)
    
    # ----- Case 3: Folder.Files ----- extract folder path
    # Check both EXPRESSION and REFERENCED_EXPRESSION for Folder.Files pattern
    expression_text = row.get('EXPRESSION', '')
    if 'Folder.Files' in expression_text:
        return extract_folder_path(expression_text)
    
    referenced_text = row.get('REFERENCED_EXPRESSION', '')
    if 'Folder.Files' in referenced_text:
        return extract_folder_path(referenced_text)
    return None

def extract_sharepoint_details(text):
    try:
        # Extract Folder Path
        folder_path_match = re.search(r'#"Folder Path"="([^"]+)"', text)
        folder_path = folder_path_match.group(1) if folder_path_match else None
        
        # Extract File Name
        name_match = re.search(r'\[Name="([^"]+)"', text)
        file_name = name_match.group(1) if name_match else None
        
        # Extract Sheet Name
        sheet_match = re.search(r'\[Item="([^"]+)",Kind="Sheet"\]', text)
        sheet_name = sheet_match.group(1) if sheet_match else None
        
        # Construct the complete file path
        if folder_path and file_name:
            complete_path = folder_path + file_name
            if sheet_name:
                complete_path += f" | Sheet: {sheet_name}"
            return complete_path
        
        return None
    except Exception as e:
        return None

def extract_folder_path(text):
    """Extract folder path from Folder.Files() pattern"""
    try:
        # Match pattern: Folder.Files("path") and extract the path
        match = re.search(r'Folder\.Files\("([^"]+)"\)', text)
        if match:
            return match.group(1)
        return None
    except Exception as e:
        return None

# Apply to the DataFrame
grouped_df['CONNECTION_STRING'] = grouped_df.apply(extract_connection_string, axis=1)

# Display selected columns
columns_to_show = ['DATABASE_NAME', 'SEMANTIC_MODEL_TABLE', 'SEMANTIC_MODEL_OBJECT', 
                   'SEMANTIC_MODEL_REFERENCED_OBJECT', 'SCHEMA', 'TABLE_OR_VIEW', 
                   'SOURCE_TYPE', 'CONNECTION_STRING','REFERENCED_OBJECT_TYPE',
                   'EXPRESSION', 'REFERENCED_EXPRESSION']

display(grouped_df[columns_to_show])

## Extract Database Names for Sources
Parsing SQL connection expressions to extract the specific database name, completing the full data lineage path from source database to semantic model object

In [ ]:
def extract_database_name(row):
    # If CONNECTION_STRING is null, return None for DATABASE_NAME
    if pd.isna(row['CONNECTION_STRING']) or row['CONNECTION_STRING'] is None:
        return None
    
    if row['SOURCE_TYPE'] == 'Database':
        text = row['REFERENCED_EXPRESSION']
        # Add additional null check for REFERENCED_EXPRESSION
        if pd.isna(text) or text is None:
            return None
        # Convert to string to handle any type issues
        text = str(text)
        if ', "' in text:
            part = text.split(', "', 1)[1]
            return part.split('"', 1)[0]
    return None

# Apply to the DataFrame
grouped_df['DATABASE'] = grouped_df.apply(extract_database_name, axis=1)

# Ensure all columns are the correct type before displaying
# This converts any problematic types to strings
columns_to_show = ['DATABASE_NAME', 'SEMANTIC_MODEL_TABLE', 'SEMANTIC_MODEL_OBJECT', 
                   'SEMANTIC_MODEL_REFERENCED_OBJECT', 'SCHEMA', 'TABLE_OR_VIEW', 
                   'SOURCE_TYPE', 'CONNECTION_STRING', 'DATABASE', 'EXPRESSION', 
                   'REFERENCED_EXPRESSION']

# Create a copy with cleaned types
display_df = grouped_df[columns_to_show].copy()

# Option 1: Convert problematic columns to string (handles None gracefully)
for col in columns_to_show:
    display_df[col] = display_df[col].astype(str).replace('None', '')

display(display_df)

# Option 2: Alternatively, just fill NaN/None values
# display_df = grouped_df[columns_to_show].fillna('')
# display(display_df)

In [ ]:
def extract_database_name(row):
    # If CONNECTION_STRING is null, return None for DATABASE_NAME
    if pd.isna(row['CONNECTION_STRING']) or row['CONNECTION_STRING'] is None:
        return None
    
    if row['SOURCE_TYPE'] == 'Database':
        text = row['REFERENCED_EXPRESSION']
        if ', "' in text:
            part = text.split(', "', 1)[1]
            return part.split('"', 1)[0]
    return None

# Apply to the DataFrame
grouped_df['DATABASE'] = grouped_df.apply(extract_database_name, axis=1)

# Display selected columns 
columns_to_show = ['DATABASE_NAME', 'SEMANTIC_MODEL_TABLE', 'SEMANTIC_MODEL_OBJECT', 'SEMANTIC_MODEL_REFERENCED_OBJECT', 'SCHEMA', 'TABLE_OR_VIEW', 'SOURCE_TYPE', 
'CONNECTION_STRING', 'DATABASE', 'EXPRESSION', 'REFERENCED_EXPRESSION']

display(grouped_df[columns_to_show])

## Save as a Table in the Lakehouse

In [ ]:
display(display_df)

In [ ]:
# Create Spark DataFrame from Pandas
spark_df = spark.createDataFrame(display_df)

# Add database name and timestamp column using Spark syntax
spark_df = spark_df.withColumn("DATABASE_TIMESTAMP", 
                                lit(f"{semantic_model_name} | {datetime.now():%d.%m.%Y %H:%M:%S}"))

# Define table name
table_name = "data_sources"

# Determine version number based on existing data
if spark.catalog.tableExists(table_name):
    # Read existing table to find the max version for this semantic model
    existing_df = spark.read.table(table_name)
    
    # Filter for current semantic model and get max version
    max_version = existing_df.filter(
        existing_df.DATABASE_VERSION.startswith(semantic_model_name)
    ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()[0]['max_ver']
    
    # Increment version, or start at 1 if this model hasn't been added yet
    next_version = (max_version + 1) if max_version is not None else 1
    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

# Add version column
from pyspark.sql.functions import lit
spark_df = spark_df.withColumn('DATABASE_VERSION', lit(f"{semantic_model_name} | V{next_version}"))

# Write using the determined mode
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

display(spark_df)

# Query and Format DAX Measures for Documentation
Extracting all measures from the semantic model, applying custom formatting to DAX expressions for improved readability, and exporting a com

In [ ]:
# Query to get all unique measures from the calcdependency table
query = f"""
SELECT DISTINCT 
    `TABLE` as TableName,
    `DATABASE_NAME` as DatabaseName,
    `OBJECT` as MeasureName,
    `EXPRESSION` as MeasureExpression
FROM calcdependency
WHERE OBJECT_TYPE = 'MEASURE'
ORDER BY 1
"""

# Read the measures from the lakehouse
measures_df = spark.sql(query).toPandas()

def format_dax(dax_expression):
    """
    Format DAX expression with proper indentation and line breaks
    """
    if pd.isna(dax_expression) or dax_expression == "":
        return dax_expression
    
    # Remove extra whitespace and normalize
    dax = re.sub(r'\s+', ' ', str(dax_expression)).strip()
    
    # Track parentheses depth for proper indentation
    result = []
    indent_level = 0
    i = 0
    current_word = ""
    in_string = False
    string_char = None
    
    # Major DAX functions that should trigger formatting
    major_functions = {
        'CALCULATE', 'CALCULATETABLE', 'FILTER', 'ALL', 'ALLEXCEPT', 'ALLSELECTED',
        'SUMX', 'AVERAGEX', 'COUNTX', 'MINX', 'MAXX', 'PRODUCTX',
        'IF', 'SWITCH', 'DIVIDE', 'FORMAT', 'CONCATENATEX',
        'DISTINCTCOUNT', 'COUNTROWS', 'VALUES', 'DISTINCT', 'COUNT', 'SUM',
        'RELATEDTABLE', 'RELATED', 'USERELATIONSHIP',
        'DATEADD', 'DATESYTD', 'TOTALYTD', 'SAMEPERIODLASTYEAR',
        'EARLIER', 'EARLIEST', 'HASONEVALUE', 'SELECTEDVALUE',
        'VAR', 'RETURN'
    }
    
    while i < len(dax):
        char = dax[i]
        
        # Handle string literals
        if char in ['"', "'"] and (i == 0 or dax[i-1] != '\\'):
            if not in_string:
                in_string = True
                string_char = char
                current_word += char
            elif char == string_char:
                in_string = False
                string_char = None
                current_word += char
            else:
                current_word += char
            i += 1
            continue
        
        if in_string:
            current_word += char
            i += 1
            continue
        
        # Handle opening parenthesis
        if char == '(':
            func_name = current_word.strip().upper()
            
            if func_name in major_functions:
                result.append(current_word.strip())
                result.append(' (')
                result.append('\n')
                indent_level += 1
                result.append('    ' * indent_level)
            else:
                result.append(current_word)
                result.append('(')
                indent_level += 1
            
            current_word = ""
            i += 1
            continue
        
        # Handle closing parenthesis
        elif char == ')':
            if current_word.strip():
                result.append(current_word.strip())
                current_word = ""
            
            indent_level = max(0, indent_level - 1)
            result.append('\n')
            result.append('    ' * indent_level)
            result.append(')')
            i += 1
            continue
        
        # Handle commas - always add line break after comma
        elif char == ',':
            result.append(current_word.strip())
            result.append(',')
            result.append('\n')
            result.append('    ' * indent_level)
            current_word = ""
            i += 1
            # Skip any whitespace after comma
            while i < len(dax) and dax[i] == ' ':
                i += 1
            continue
        
        # Handle VAR keyword specially
        elif current_word.strip().upper() == 'VAR' and char == ' ':
            result.append('\n')
            result.append('VAR ')
            current_word = ""
            i += 1
            continue
        
        # Handle RETURN keyword specially
        elif current_word.strip().upper() == 'RETURN' and char == ' ':
            result.append('\n')
            result.append('RETURN')
            result.append('\n')
            result.append('    ')
            current_word = ""
            i += 1
            continue
        
        else:
            current_word += char
            i += 1
    
    # Add any remaining content
    if current_word.strip():
        result.append(current_word.strip())
    
    # Join and clean up
    formatted = ''.join(result)
    
    # Clean up excessive blank lines
    lines = formatted.split('\n')
    cleaned_lines = []
    prev_blank = False
    
    for line in lines:
        if line.strip() == '':
            if not prev_blank:
                cleaned_lines.append(line)
            prev_blank = True
        else:
            cleaned_lines.append(line)
            prev_blank = False
    
    formatted = '\n'.join(cleaned_lines)
    
    # Remove leading/trailing whitespace
    formatted = formatted.strip()
    
    return formatted

# Apply formatting to the MeasureExpression column
measures_df['FormattedExpression'] = measures_df['MeasureExpression'].apply(format_dax)

# Keep only the required columns
measures_df = measures_df[['TableName','DatabaseName','MeasureName', 'FormattedExpression']]

# Set display options to show full content
pd.set_option('display.max_colwidth', None)

print(f"Total measures found: {len(measures_df)}")

#display(measures_df[[DatabaseName'MeasureName', 'FormattedExpression']])
display(measures_df)

# Setup Data Agent & Generate the AI-Generated Documentation

## Define Helper Function for Message Display
Creating a utility function to format and display conversational messages from the data agent in a readable format, showing role and content for each interaction

In [ ]:
from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent
)

## Define Helper Function for Message Display
Creating a utility function to format and display conversational messages from the data agent in a readable format, showing role and content for each interaction

In [ ]:
# Pretty printing helper
def pretty_print(messages):
    print("Messages")
    for m in messages:
        print(f"{m.role}: {m.content[0].text.value}")
    print()

## Initialize Fabric Data Agent
Creating a new data agent instance for the OrderIncome semantic model or connecting to an existing agent to enable natural language querying and AI-powered data exploration

In [ ]:
import time
from fabric.dataagent.client import create_data_agent, delete_data_agent

# Delete the existing agent first
try:
    delete_data_agent(data_agent_name)
    print(f"Existing agent '{data_agent_name}' deleted")
except Exception as e:
    print(f"No existing agent to delete or deletion failed: {e}")

# Wait a moment after deletion
time.sleep(5)

# Now create the new one with retry logic
max_retries = 5
retry_delay = 30  # seconds

for attempt in range(max_retries):
    try:
        data_agent = create_data_agent(data_agent_name)
        print(f"Successfully created agent: {data_agent_name}")
        break
    except Exception as e:
        if "not available yet" in str(e).lower() or "already in use" in str(e).lower():
            if attempt < max_retries - 1:
                print(f"Agent name not available yet. Waiting {retry_delay} seconds... (Attempt {attempt + 1}/{max_retries})")
                time.sleep(retry_delay)
            else:
                print("Max retries reached. Please wait a few minutes and try again.")
                raise
        else:
            print(f"Unexpected error: {e}")
            raise e

## Configure Data Agent with Domain-Specific Instructions
Setting up the agent with specialized knowledge about Power BI semantic models, DAX dependency analysis, and the structure of the CALCDEPENDENCY function output to enable accurate interpretation of direct and indirect dependencies between model objects

In [ ]:
data_agent.update_configuration(
    instructions = f"You are a Power BI semantic model developer.\
The {semantic_model_name}_calcdependency table contains the output of the DAX {dax_function} function\
for another Power BI semantic model. It contains information about the dependencies between the columns, tables, measures, \
calculated columns, calculated tables and relationships in that model. \
Each row in the {semantic_model_name}_calcdependency table represents a dependency between two objects. \
The combination of values in the OBJECT_TYPE, TABLE and OBJECT column uniquely identifies the object \
which is the source of the dependency. \
The combination of values in the REFERENCED_OBJECT_TYPE, REFERENCED_TABLE and _REFERENCED_OBJECT columns \
uniquely identifies the object which is the target of the identity. \
You can join the table to itself multiple times to find chains of dependencies between objects. \
When I ask about dependencies, please include direct dependencies and also indirect dependencies, \
for example where one object has a dependency on another object which in turn has a dependency on the object I am asking about. \
The EXPRESSION column contains the DAX definition of an object.\
A value of CALC_COLUMN in the OBJECT_TYPE table indicates that the object is a calculated column.\
Provide only the 'Measure Description', 'How the measure works', and 'The Measure Results'. Provide some sample values for 'The Measure Results' if possible.\
Do not mention the measure name.\
Do not discuss or reveal any sensitive data including political topics, personal information, or confidential business data."
)
data_agent.get_configuration()

## Connect Data Agent to Lakehouse Data Source
Adding the OrderIncome lakehouse as a data source for the agent, enabling it to query and analyze the semantic model dependency data stored in the lakehouse tables

In [ ]:
# add a lakehouse
#datasource type could be: lakehouse, kqldatabase, datawarehouse or semanticmodel
data_agent.add_datasource(lakehouse_name, type="lakehouse")
#data_agent.remove_datasource(lakehouse_name)

## Select Target Table for Agent Analysis
Configuring the agent to focus on the orderincome_calcdependency table within the dbo schema, which contains the DAX dependency relationships and expressions for analysis

In [ ]:
# Retrieve the data source object (assumes one was added)
datasource = data_agent.get_datasources()[0]

# Select relevant tables from the Lakehouse (schema: dbo)
datasource_table = f"calcdependency"
datasource.select(datasource_table)


## Display Data Source Configuration
Outputting the current data source configuration to verify the connected lakehouse, selected schema, and table setup before executing queries

## Publish Data Agent Configuration
Deploying the configured data agent with its instructions, data sources, and table selections, making it ready to receive and process natural language queries

In [ ]:
#publish the agent
data_agent.publish()

## Create OpenAI Assistant and Conversation Thread
Initializing an OpenAI GPT-4o assistant connected to the Fabric data agent and creating a new conversation thread to enable interactive natural language querying of the semantic model dependencies

In [ ]:
#import sempy.fabric as fabric
#from fabric.dataagent.client import FabricOpenAI

fabric_client = FabricOpenAI(artifact_name=data_agent_name)
assistant = fabric_client.beta.assistants.create(model=model_name)
thread = fabric_client.beta.threads.create()

print(assistant.id)
print(thread.id)

## Generate AI-Powered Documentation for DAX Measures
Iterating through all measures in batches, sending each measure name and formatted expression to the AI agent, and collecting comprehensive documentation that explains what each measure calculates and how it works, with built-in error handling and rate limiting

In [ ]:
# Define batch size
batch_size = 1
total_measures = len(measures_df)
all_documentation = []

# Loop through measures in batches
for i in range(0, total_measures, batch_size):
    batch = measures_df.iloc[i:i+batch_size]
    
    # Extract both MeasureName and FormattedExpression
    measure_info = []
    for idx, row in batch.iterrows():
        measure_info.append({
            'name': row['MeasureName'],
            'expression': row['FormattedExpression'],
            'database': row['DatabaseName'],
            'table': row['TableName']
        })
    
    # Create formatted string for the prompt
    measures_list = ", ".join([f"{m['name']}" for m in measure_info])
    measures_with_expressions = "\n".join([
        f"- {m['name']}: {m['expression']}" for m in measure_info
    ])
    
    print(f"\n{'='*60}")
    print(f"Processing batch {i//batch_size + 1}: Measures {i+1} to {min(i+batch_size, total_measures)}")
    print(f"Measures: {measures_list}")
    print('='*60)
    
    # Create a message for this batch with both name and expression
    fabric_client.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=f"Generate documentation for these measures:\n{measures_with_expressions}\n\nFor each measure, describe what it calculates and how it works."
    )
    
    # Create and run the request
    run = fabric_client.beta.threads.runs.create(
        thread_id=thread.id, 
        assistant_id=assistant.id
    )
    
    # Wait for completion
    while run.status != "completed":
        run = fabric_client.beta.threads.runs.retrieve(
            thread_id=thread.id, 
            run_id=run.id
        )
        time.sleep(2)
        
        # Handle potential failures
        if run.status in ["failed", "cancelled", "expired"]:
            print(f"Run failed with status: {run.status}")
            break
    
    # Retrieve the response
    messages = fabric_client.beta.threads.messages.list(
        thread_id=thread.id, 
        order="desc",
        limit=1
    )
    
    # Get the latest assistant message
    if messages.data:
        response_text = messages.data[0].content[0].text.value
        print("\nAgent Response:")
        print(response_text)
        
        # Store documentation with both fields
        all_documentation.append({
            'batch': i//batch_size + 1,
            'measure_name': measure_info[0]['name'] if len(measure_info) == 1 else [m['name'] for m in measure_info],
            'database_name': measure_info[0]['database'] if len(measure_info) == 1 else [m['database'] for m in measure_info],
            'table_name': measure_info[0]['table'] if len(measure_info) == 1 else [m['table'] for m in measure_info],
            'formatted_expression': measure_info[0]['expression'] if len(measure_info) == 1 else [m['expression'] for m in measure_info],
            'measures': measures_list,
            'documentation': response_text
        })
    
    # Small delay between batches to avoid rate limiting
    time.sleep(3)

print(f"\n{'='*60}")
print(f"Completed documentation for {total_measures} measures in {len(all_documentation)} batches")
print('='*60)



In [ ]:
display(all_documentation)

In [ ]:
#newly added

import pandas as pd
import re

#'Measure Description', 'How the measure works', and 'The Measure Results'

def clean_text(text):
    """Remove common markdown and documentation symbols from text."""
    if not text:
        return text
    
    # Remove markdown bold/italic markers
    text = re.sub(r'\*\*|\*|__|_', '', text)
    
    # Remove forward slashes (// or standalone /)
    text = re.sub(r'/{1,}', '', text)
    
    # Remove backslashes
    text = re.sub(r'\\+', '', text)
    
    # Remove markdown headers (##, ###, etc.)
    text = re.sub(r'^#{1,6}\s*', '', text, flags=re.MULTILINE)
    
    # Remove HTML tags if any
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove bullet point symbols (-, •, *, >, ~)
    text = re.sub(r'^\s*[-•>~]\s+', '', text, flags=re.MULTILINE)
    
    # Remove excessive whitespace/newlines
    text = re.sub(r'\n{3,}', '\n\n', text)   # max 2 consecutive newlines
    text = re.sub(r'[ \t]{2,}', ' ', text)   # max 1 space
    
    # Final strip
    return text.strip()


def parse_documentation(doc_text):
    """
    Parse documentation text into Description, How the measure works, and Result sections.
    """
    if pd.isna(doc_text) or doc_text == '':
        return pd.Series({
            'description': None,
            'how_the_measure_works': None,
            'result': None
        })
    
    # Initialize sections
    description = ''
    how_it_works = ''
    result = ''
    
    # Split by common headers (case-insensitive)
    desc_pattern = r'(?:__|\\*\\*)?Measure Description(?:__|\\*\\*)?:?\s*'
    how_pattern = r'(?:__|\\*\\*)?How the measure works(?:__|\\*\\*)?:?\s*'
    result_pattern = r'(?:__|\\*\\*)?The Measure Results(?:__|\\*\\*)?:?\s*'
    
    # Find section positions
    desc_match = re.search(desc_pattern, doc_text, re.IGNORECASE)
    how_match = re.search(how_pattern, doc_text, re.IGNORECASE)
    result_match = re.search(result_pattern, doc_text, re.IGNORECASE)
    
    # Extract Description
    if desc_match and how_match:
        description = doc_text[desc_match.end():how_match.start()].strip()
    elif desc_match:
        if result_match:
            description = doc_text[desc_match.end():result_match.start()].strip()
        else:
            description = doc_text[desc_match.end():].strip()
    
    # Extract How the measure works
    if how_match and result_match:
        how_it_works = doc_text[how_match.end():result_match.start()].strip()
    elif how_match:
        how_it_works = doc_text[how_match.end():].strip()
    
    # Extract Result
    if result_match:
        result = doc_text[result_match.end():].strip()
    
    # Apply cleaning to each section before returning
    return pd.Series({
        'description': clean_text(description) if description else None,
        'how_the_measure_works': clean_text(how_it_works) if how_it_works else None,
        'result': clean_text(result) if result else None
    })


# Convert all_documentation to DataFrame
documentation_df = pd.DataFrame(all_documentation)

# Apply the parsing function to the 'documentation' column
parsed_sections = documentation_df['documentation'].apply(parse_documentation)

# Combine with original dataframe (keeping the 'documentation' column)
final_df = pd.concat([documentation_df, parsed_sections], axis=1)

# Quick audit - see all unique non-alphanumeric characters remaining in your text
import collections
all_chars = collections.Counter(
    c for text in final_df['description'].dropna() 
    for c in text if not c.isalnum() and not c.isspace()
)
print("Remaining special characters:", all_chars.most_common(20))

# Display the result
display(final_df)

In [ ]:
import pandas as pd
import re

#'Measure Description', 'How the measure works', and 'The Measure Results'

def parse_documentation(doc_text):
    """
    Parse documentation text into Description, How the measure works, and Result sections.
    """
    if pd.isna(doc_text) or doc_text == '':
        return pd.Series({
            'description': None,
            'how_the_measure_works': None,
            'result': None
        })
    
    # Initialize sections
    description = ''
    how_it_works = ''
    result = ''
    
    # Split by common headers (case-insensitive)
    desc_pattern = r'(?:__|\\*\\*)?Measure Description(?:__|\\*\\*)?:?\s*'
    how_pattern = r'(?:__|\\*\\*)?How the measure works(?:__|\\*\\*)?:?\s*'
    result_pattern = r'(?:__|\\*\\*)?The Measure Results(?:__|\\*\\*)?:?\s*'
    
    # Find section positions
    desc_match = re.search(desc_pattern, doc_text, re.IGNORECASE)
    how_match = re.search(how_pattern, doc_text, re.IGNORECASE)
    result_match = re.search(result_pattern, doc_text, re.IGNORECASE)
    
    # Extract Description
    if desc_match and how_match:
        description = doc_text[desc_match.end():how_match.start()].strip()
    elif desc_match:
        if result_match:
            description = doc_text[desc_match.end():result_match.start()].strip()
        else:
            description = doc_text[desc_match.end():].strip()
    
    # Extract How the measure works
    if how_match and result_match:
        how_it_works = doc_text[how_match.end():result_match.start()].strip()
    elif how_match:
        how_it_works = doc_text[how_match.end():].strip()
    
    # Extract Result
    if result_match:
        result = doc_text[result_match.end():].strip()
    
    return pd.Series({
        'description': description if description else None,
        'how_the_measure_works': how_it_works if how_it_works else None,
        'result': result if result else None
    })

# Convert all_documentation to DataFrame
documentation_df = pd.DataFrame(all_documentation)

# Apply the parsing function to the 'documentation' column
parsed_sections = documentation_df['documentation'].apply(parse_documentation)

# Combine with original dataframe (keeping the 'documentation' column)
final_df = pd.concat([documentation_df, parsed_sections], axis=1)

# Display the result
display(final_df)

## Save the DAX Documentation Table

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

# First, create the DataFrame from all_documentation
documentation_df = pd.DataFrame(final_df)

# Rename columns to match the expected uppercase names
documentation_df = documentation_df.rename(columns={
    'database_name': 'DATABASE_NAME',
    'batch': 'BATCH',
    'measure_name': 'MEASURE_NAME',
    'table_name': 'TABLE_NAME',
    'formatted_expression': 'FORMATTED_EXPRESSION',
    'documentation': 'AI_DOCUMENTATION',
    'description': 'DESCRIPTION',
    'how_the_measure_works': 'HOW_THE_MEASURE_WORKS',
    'result': 'RESULT'
})

# Select final columns
documentation_df = documentation_df[
    [
        'DATABASE_NAME',
        'BATCH',
        'MEASURE_NAME',
        'TABLE_NAME',
        'FORMATTED_EXPRESSION',
        'AI_DOCUMENTATION',
        'DESCRIPTION',
        'HOW_THE_MEASURE_WORKS',
        'RESULT'
    ]
]

# Define explicit schema to avoid void types
schema = StructType([
    StructField("DATABASE_NAME", StringType(), True),
    StructField("BATCH", StringType(), True),
    StructField("MEASURE_NAME", StringType(), True),
    StructField("TABLE_NAME", StringType(), True),
    StructField("FORMATTED_EXPRESSION", StringType(), True),
    StructField("AI_DOCUMENTATION", StringType(), True),
    StructField("DESCRIPTION", StringType(), True),
    StructField("HOW_THE_MEASURE_WORKS", StringType(), True),
    StructField("RESULT", StringType(), True)
])

# Create Spark DataFrame with explicit schema
spark_df = spark.createDataFrame(documentation_df, schema=schema)

# Add database name and timestamp column using Spark syntax
spark_df = spark_df.withColumn("DATABASE_TIMESTAMP", 
                                lit(f"{semantic_model_name} | {datetime.now():%d.%m.%Y %H:%M:%S}"))

# Define table name
table_name = "agent_dax_documentation"

# Determine version number based on existing data
if spark.catalog.tableExists(table_name):
    # Read existing table to find the max version for this semantic model
    existing_df = spark.read.table(table_name)
    
    # Filter for current semantic model and get max version
    max_version = existing_df.filter(
        existing_df.DATABASE_VERSION.startswith(semantic_model_name)
    ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()[0]['max_ver']
    
    # Increment version, or start at 1 if this model hasn't been added yet
    next_version = (max_version + 1) if max_version is not None else 1
    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

# Add version column
from pyspark.sql.functions import lit
spark_df = spark_df.withColumn('DATABASE_VERSION', lit(f"{semantic_model_name} | V{next_version}"))

# Write using the determined mode
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

display(spark_df)

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.agent_dax_documentation LIMIT 1000")
display(df)